In [1]:
# =================================================================================================
# Raw data : NHANES 2003-2014 Demographics, Laboratory, and Questionnaire datasets
# Purpose : Create one unified participant-level NHANES dataset by aggregating all domains using SEQN.
# Specification : Each NHANES dataset has SEQN "Participant ID", which is the unique ID used for merge.
# Files Used :
#   neighbourhood_nhanes_demographics_2003_2014_final.csv
#   neighbourhood_nhanes_labs_2003_2014_appended.csv
#   neighbourhood_nhanes_questionnaire_2003_2014_appended.csv
# Data Dictionary : NHANES 2003-2014 documentation
# Data Availability Matrix : Generated in this notebook using domain presence flags
# Source Site : https://wwwn.cdc.gov/nchs/nhanes/
# Developer Notes : Full outer join is used so no SEQN is dropped from any file.
# File Creator : Neha Khairnar
# =================================================================================================

# 📦 Step 1: Imports
import pandas as pd
from pathlib import Path

# =================================================================================================
# 📂 Step 2: Define Paths
# =================================================================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data" / "processed_v3" / "2011_2012"
demo_file = DATA_DIR / "neighbourhood_nhanes_demographics_2011_2012_raw_corrected.csv"
labs_file = DATA_DIR / "neighbourhood_nhanes_labs_2011_2012_raw_corrected.csv"
questionnaire_file = DATA_DIR / "neighbourhood_nhanes_questionnaire_2011_2012_raw_corrected.csv"
Examination_file = DATA_DIR / "neighbourhood_nhanes_examination_2011_2012_raw_corrected.csv"
Dietary_file = DATA_DIR / "neighbourhood_nhanes_dietary_2011_2012_raw_corrected.csv"


# =================================================================================================
# 📥 Step 3: Load Files
# =================================================================================================

demo = pd.read_csv(demo_file, dtype=str, keep_default_na=False)
labs = pd.read_csv(labs_file, dtype=str, keep_default_na=False)
questionnaire = pd.read_csv(questionnaire_file, dtype=str, keep_default_na=False)
Examination = pd.read_csv(Examination_file, dtype=str, keep_default_na=False)
Dietary = pd.read_csv(Dietary_file, dtype=str, keep_default_na=False)

print("📊 Loaded Files:")
print("Demographics:", demo.shape)
print("Labs:", labs.shape)
print("Questionnaire:", questionnaire.shape)
print("Examination:", Examination.shape)
print("Dietary:", Dietary.shape)

# =================================================================================================
# 🧹 Step 4: Standardize Column Names
# =================================================================================================
demo.columns = demo.columns.str.upper()
labs.columns = labs.columns.str.upper()
questionnaire.columns = questionnaire.columns.str.upper()
Examination.columns = Examination.columns.str.upper()
Dietary.columns = Dietary.columns.str.upper()

# =================================================================================================
# ✅ Step 5: Validate SEQN Exists
# =================================================================================================

for name, df_check in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire,
    "Examination":Examination,
    "Dietary":Dietary
}.items():
    if "SEQN" not in df_check.columns:
        raise ValueError(f"❌ SEQN column missing in {name} file")
    else:
        print(f"✅ SEQN found in {name}")

# =================================================================================================
# 📊 Step 6: File Structure Summary
# =================================================================================================
summary = []
for name, df_temp in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire,
    "Examination":Examination,
    "Dietary":Dietary
}.items():
    summary.append({
        "file": name,
        "rows": df_temp.shape[0],
        "columns": df_temp.shape[1],
        "unique_seqn": df_temp["SEQN"].nunique(),
        "duplicate_seqn_count": df_temp.duplicated(subset=["SEQN"]).sum(),
        "column_names": list(df_temp.columns)
    })

summary_df = pd.DataFrame(summary)
display(summary_df[["file", "rows", "columns", "unique_seqn", "duplicate_seqn_count"]])

final_df = (
    demo
    .merge(questionnaire, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_QUESTIONNAIRE"))
    .merge(labs, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_LAB"))
    .merge(Examination, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_EXAMINATION"))
    .merge(Dietary, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_DIETARY"))
)
# =================================================================================================
# 👀 Step 10: Validate Final Unified Dataset
# =================================================================================================

print("📊 Final Unified Dataset Shape:", final_df.shape)
print("Unique SEQN Count:", final_df["SEQN"].nunique())
print("Duplicate SEQN Count:", final_df.duplicated(subset=["SEQN"]).sum())
display(final_df.head(20))

OUTPUT_DIR = BASE_DIR / "data" / "processed_v3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_unified_2011_2012_corrected.csv"
final_df.to_csv(OUTPUT_PATH, index=False)
print("\n✅ Final unified NHANES dataset saved to:")
print(OUTPUT_PATH)


📊 Loaded Files:
Demographics: (9756, 50)
Labs: (8956, 35)
Questionnaire: (9756, 333)
Examination: (9338, 26)
Dietary: (9338, 244)
✅ SEQN found in demographics
✅ SEQN found in labs
✅ SEQN found in questionnaire
✅ SEQN found in Examination
✅ SEQN found in Dietary


,file,rows,columns,unique_seqn,duplicate_seqn_count
0,demographics,9756,50,9756,0
1,labs,8956,35,8956,0
2,questionnaire,9756,333,9756,0
3,Examination,9338,26,9338,0
4,Dietary,9338,244,9338,0


📊 Final Unified Dataset Shape: (9756, 684)
Unique SEQN Count: 9756
Duplicate SEQN Count: 0


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGY,...,DR2TP184,DR2TP204,DR2TP205,DR2TP225,DR2TP226,DR2_300,DR2_320Z,DR2_330Z,DR2BWATZ,DR2TWS
0,62161,7,2,1,22,,3,3,2,,...,0.002,0.20700000000000002,0.011,0.016,0.013000000000000001,1,0.0,0.0,0.0,1
1,62162,7,2,2,3,,1,1,1,3,...,0.001,0.046,0.005,0.005,0.003,2,118.5,0.0,118.5,1
2,62163,7,2,1,14,,5,6,2,14,...,0.001,0.07300000000000001,0.003,0.025,0.001,2,429.56,429.56,0.0,99
3,62164,7,2,2,44,,3,3,1,,...,0.0,0.136,0.008,0.013,0.01,2,0.0,0.0,0.0,1
4,62165,7,2,2,14,,4,4,2,14,...,0.003,0.134,0.006,0.013,0.009999999999999998,2,222.19,0.0,222.19,4
5,62166,7,2,1,9,,3,3,2,10,...,,,,,,,,,,
6,62167,7,2,1,0,11,5,6,1,,...,0.0,0.153,0.002,0.008,0.066,2,311.06,311.06,0.0,99
7,62168,7,2,1,6,,5,7,1,6,...,,,,,,,,,,
8,62169,7,2,1,21,,5,6,1,,...,0.005,0.096,0.027,0.036000000000000004,0.114,2,1018.44,518.44,500.0,4
9,62170,7,2,1,15,,5,7,1,15,...,0.029,0.027,0.022,0.001,0.02,2,1593.98,1593.98,0.0,1



✅ Final unified NHANES dataset saved to:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\processed_v3\neighbourhood_nhanes_unified_2011_2012_corrected.csv
